# DB structure and basic queries

This notebook uses native sqlite3 to efficiently query the large Spotify databases.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import random
from autoencoder.db import get_connection
from autoencoder.db import get_tables
from autoencoder.db import get_columns
from autoencoder.db import head_table


DATABASE_PATH = Path.home() / "HDD/Datasets/annas_archive_spotify_2025_07_merged/spotify_merged.sqlite3"

## Connect to Databases

Create separate connections to each database. Connections are instant with native sqlite3.

In [2]:
conn  = get_connection(DATABASE_PATH)
print("Connected to database")

Connected to database


## Explore Schema

List tables and their columns in each database.

In [3]:
for table in get_tables(conn):
    columns = get_columns(conn, table)
    col_str = ", ".join(f"{name} ({dtype})" for name, dtype in columns)
    print(f"  {table}: {col_str}")

  album_images: album_rowid (INTEGER), width (INTEGER), height (INTEGER), url (TEXT)
  albums: rowid (INTEGER), id (TEXT), fetched_at (INTEGER), name (TEXT), album_type (TEXT), available_markets_rowid (INTEGER), external_id_upc (TEXT), copyright_c (TEXT), copyright_p (TEXT), label (TEXT), popularity (INTEGER), release_date (TEXT), release_date_precision (TEXT), total_tracks (INTEGER), external_id_amgid (TEXT)
  artist_albums: artist_rowid (INTEGER), album_rowid (INTEGER), is_appears_on (INTEGER), is_implicit_appears_on (INTEGER), index_in_album (INTEGER)
  artist_genres: artist_rowid (INTEGER), genre (TEXT)
  artist_images: artist_rowid (INTEGER), width (INTEGER), height (INTEGER), url (TEXT)
  artists: rowid (INTEGER), id (TEXT), fetched_at (INTEGER), name (TEXT), followers_total (INTEGER), popularity (INTEGER)
  audio_features: track_rowid (INT), track_id (TEXT), time_signature (INT), tempo (INT), key (INT), mode (INT), danceability (REAL), energy (REAL), loudness (REAL), speechiness

## Sample Data

Fetch random samples from key tables.

In [4]:
def sample_table(
    conn: sqlite3.Connection,
    table: str,
    n: int = 1000,
) -> pd.DataFrame:
    """
    Get a random sample from a table using ROWID-based sampling.

    This is fast even on huge tables because it:
    1. Gets max ROWID (uses index, instant)
    2. Generates random ROWIDs in Python
    3. Fetches only those specific rows
    """
    # Get max rowid
    cursor = conn.execute(f"SELECT MAX(rowid) FROM {table}")
    max_rowid = cursor.fetchone()[0]

    if max_rowid is None or max_rowid == 0:
        return pd.DataFrame()

    # Generate random rowids (sample more than needed to account for gaps)
    sample_size = min(n * 3, max_rowid)
    random_rowids = random.sample(range(1, max_rowid + 1), sample_size)

    # Fetch rows with those rowids
    placeholders = ",".join(["?"] * len(random_rowids))
    query = f"SELECT * FROM {table} WHERE rowid IN ({placeholders}) LIMIT {n}"
    return pd.read_sql_query(query, conn, params=random_rowids)


In [5]:
# Sample tracks
tracks_sample = sample_table(conn, "tracks", n=5)
tracks_sample

,rowid,id,fetched_at,name,preview_url,album_rowid,track_number,external_id_isrc,popularity,available_markets_rowid,disc_number,duration_ms,explicit
0,5976946,6dKelj9BLIFAL24cJ55vjz,1744243200000,Flycatchers,https://p.scdn.co/mp3-preview/dcecf0620b06057a...,490469,81,NL3WF2009881,0,1,1,100918,0
1,12417439,14C53tDp9LjvhnTGjqC9G3,1742428800000,Midnight's Dance Among Leaves,None,1514914,16,UKZVC3305474,0,1,1,107000,0
2,20810374,3qY96yZRT9GnFrkcBwYKXq,1744243200000,Tranquil Stream Work,https://p.scdn.co/mp3-preview/97754b1923fac63d...,2093732,3,QZZEB2390743,0,3,1,120000,0
3,32214990,7tFrmZqPUUGquw4E7PXHxW,1744243200000,Dixie,None,3184726,2,GBLPF0277893,0,126,1,163329,0
4,42720786,3HqZvWGL1hR50vg80ay0dh,1743033600000,Palomitas de Maiz,https://p.scdn.co/mp3-preview/4b588377497dfa2e...,5324670,18,QZNJY2146921,1,3,1,253701,0


In [6]:
# Sample artist-track connections
tracks_artists_sample = sample_table(conn, "track_artists", n=5)
tracks_artists_sample

,track_rowid,artist_rowid
0,457219,6818373
1,3336922,522007
2,24970007,2045922
3,63655007,1187502
4,96344153,2767828


In [7]:
# Sample audio features
audio_sample = sample_table(conn, "audio_features", n=5)
audio_sample

,track_rowid,track_id,time_signature,tempo,key,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
0,182403327,3VhCU6z7BOqxwDDKPW8O8G,4,46.977,0,1,0.149,0.2550,-11.454,0.0372,0.8680,0.000021,0.4220,0.0843
1,166614269,5wCJbxZKNfAkiLhMANwWJq,4,70.860,4,0,0.691,0.0189,-25.065,0.0676,0.9830,0.938000,0.1080,0.2040
2,200702711,1ursU9xDqQBOXGJAHxbjC9,4,105.829,4,0,0.828,0.2910,-14.020,0.0487,0.7220,0.810000,0.0803,0.4140
3,214931447,7m2Ra7IGz8k3HoHo7WKiQW,4,129.925,7,1,0.712,0.3000,-11.191,0.0382,0.6170,0.000007,0.1040,0.1960
4,231386468,6e65b385n91Y6Mp7HCgpbG,4,77.006,9,0,0.577,0.9430,-5.137,0.0661,0.0116,0.000223,0.1560,0.7490


In [8]:
# Sample artists
artists_sample = sample_table(conn, "artists", n=5)
artists_sample

,rowid,id,fetched_at,name,followers_total,popularity
0,1136415,6WPpJVNZHbomX7Iu9hL7Y8,1743033600000,OG7even,3,0
1,2297371,5IwIx08MreXRPLHrQPWOMp,1743033600000,Евгения Белоусова,76,9
2,3607269,3qNCJ94OBsoG2Ft5yiQ0ZR,1743033600000,Carlos Pussetto,1,0
3,4631087,2hm9Do6e3qvhE5fe9ci3qh,1743033600000,Matias Eliel,16,0
4,6470823,0eamBqWG1mcNzR0Lwbrl63,1743033600000,CJ Kaamiño,125,3


## Query Tracks with Audio Features

Since we have separate databases, we fetch track IDs first, then query audio features.

In [9]:
tracks = pd.read_sql(
    "SELECT id, name, popularity, duration_ms FROM tracks LIMIT 5",
    conn,
)
tracks.describe()

track_ids_str = ", ".join(tracks["id"].map(lambda x: f"'{x}'"))
audio_features = pd.read_sql(
    f"SELECT * FROM audio_features WHERE track_id IN ({track_ids_str})",
    conn,
)

tracks_with_features = tracks.merge(audio_features, left_on="id", right_on="track_id")
tracks_with_features

,id,name,popularity,duration_ms,track_rowid,track_id,time_signature,tempo,key,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
0,5xHgo5JN0wfsV41HnRaos5,The Giver,89,202768,1,5xHgo5JN0wfsV41HnRaos5,4,93.024,4,1,0.558,0.901,-3.208,0.0583,0.07310,0.000000,0.3480,0.678
1,4kcRyBdCtBxcq14yDzVjJ0,Another Life,43,139388,2,4kcRyBdCtBxcq14yDzVjJ0,4,150.013,4,1,0.630,0.887,-5.604,0.0718,0.00341,0.440000,0.2260,0.912
2,2x1TExAkrUFFxVk616CKw8,Lover Online,43,167444,3,2x1TExAkrUFFxVk616CKw8,4,131.961,11,1,0.768,0.844,-5.247,0.0949,0.09800,0.103000,0.1940,0.631
3,0Hf3fR6XKINmMB4Fey7TiH,Crash,56,147601,4,0Hf3fR6XKINmMB4Fey7TiH,4,132.078,8,1,0.740,0.866,-7.083,0.2730,0.51400,0.000035,0.2810,0.500
4,0nWMjJ0b226HZP139cPKqw,I Just Missed A Call,51,146424,5,0nWMjJ0b226HZP139cPKqw,4,133.022,2,1,0.693,0.727,-7.868,0.0911,0.55100,0.282000,0.0988,0.702


In [10]:
conn.close()